# 하이퍼파라미터 튜닝 v2
- **확정**: vol3d + ID+FIN+MKT+EMO / ±0.7% / depth=3 고정
- **추가 탐색**: max_iter (트리 개수) × l2_regularization
- **탐색 조합**: lr(3) × leaf(3) × max_iter(3) × l2_reg(3) = 81가지
- **규칙**: CV로만 최적 선택 → holdout 단 1회

In [1]:
import pandas as pd
import numpy as np
import itertools
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report

In [2]:
# ── 설정 ──────────────────────────────────────────────
DATA_DIR     = './'
HOLDOUT_DAYS = 8
N_SPLITS     = 5
LABEL_COL    = 'label_07'

FIN_COLS = [
    'return_1d','return_3d','return_5d','volatility_3d',
    'volume_change','volume_ma_ratio','sector_return_mean',
    'relative_return_to_sector','relative_return_to_market',
    'relative_volatility_to_sector_3d'
]
MKT_COLS = [
    'VIX_Close','VIX_return_1d','VIX_change_3d',
    'SPY_return_1d','SPY_return_3d','QQQ_return_3d',
    'Oil_return_1d','Gold_return_1d','Dollar_return_1d','Treasury10Y_return_1d'
]

# ── 탐색 그리드 (depth=3 고정) ────────────────────────
PARAM_GRID = {
    'learning_rate'    : [0.01, 0.03, 0.05],
    'min_samples_leaf' : [20, 50, 100],
    'max_iter'         : [100, 200, 300],
    'l2_regularization': [0.0, 0.5, 1.0],
}
MAX_DEPTH = 3

n_combos = 1
for v in PARAM_GRID.values():
    n_combos *= len(v)
print(f'탐색 조합 수: {n_combos}가지  (depth={MAX_DEPTH} 고정)')

탐색 조합 수: 81가지  (depth=3 고정)


In [4]:
# ── 데이터 로드 및 라벨 계산 ──────────────────────────
raw = pd.read_csv(DATA_DIR + 'israel_hamas_FIN_MKT_features(step3).csv')
raw.columns = raw.columns.str.strip()
raw['Date'] = pd.to_datetime(raw['Date'])
raw = raw.sort_values(['ticker','Date']).reset_index(drop=True)
raw['next_return'] = raw.groupby('ticker')['Close'].pct_change(1).shift(-1)

raw['label_07'] = np.nan
raw.loc[raw['next_return'] >  0.007, 'label_07'] = 2
raw.loc[raw['next_return'] < -0.007, 'label_07'] = 0
raw.loc[(raw['next_return'] >= -0.007) & (raw['next_return'] <= 0.007), 'label_07'] = 1
labels = raw[['Date','ticker','label_07']]

df = pd.read_csv(DATA_DIR + 'model_input_dataset_3day_volatility(step3).csv', parse_dates=['Date'])
df = pd.merge(df, labels, on=['Date','ticker'], how='left')

emo_cols = [c for c in df.columns if c.startswith('emo_')]
id_cols  = [c for c in df.columns if (c.startswith('sector_') or c.startswith('ticker_')) and c != 'sector_return_mean']
FEAT_COLS = id_cols + FIN_COLS + MKT_COLS + emo_cols
print(f'피처 수: {len(FEAT_COLS)}  (ID:{len(id_cols)} FIN:{len(FIN_COLS)} MKT:{len(MKT_COLS)} EMO:{len(emo_cols)})')

피처 수: 157  (ID:125 FIN:10 MKT:10 EMO:12)


In [5]:
# ── Train / Holdout 분리 ──────────────────────────────
sub = df[FEAT_COLS + [LABEL_COL, 'Date']].dropna(subset=[LABEL_COL])

dates         = sorted(sub['Date'].unique())
holdout_dates = dates[-HOLDOUT_DAYS:]
train_dates   = dates[:-HOLDOUT_DAYS]

train = sub[sub['Date'].isin(train_dates)].copy()
hold  = sub[sub['Date'].isin(holdout_dates)].copy()

X_tr = train[FEAT_COLS].values
y_tr = train[LABEL_COL].values.astype(int)
X_ho = hold[FEAT_COLS].values
y_ho = hold[LABEL_COL].values.astype(int)

date_to_idx = {d: i for i, d in enumerate(train_dates)}
groups = train['Date'].map(date_to_idx).values

print(f'Train: {len(train)}행  ({train_dates[0].date()} ~ {train_dates[-1].date()})')
print(f'Holdout: {len(hold)}행  ({holdout_dates[0].date()} ~ {holdout_dates[-1].date()})')

Train: 4165행  (2023-09-07 ~ 2023-10-25)
Holdout: 952행  (2023-10-26 ~ 2023-11-06)


In [ ]:
# ── CV 그리드서치 ─────────────────────────────────────
gkf = GroupKFold(n_splits=N_SPLITS)
results = []

for lr, leaf, n_iter, l2 in itertools.product(
    PARAM_GRID['learning_rate'],
    PARAM_GRID['min_samples_leaf'],
    PARAM_GRID['max_iter'],
    PARAM_GRID['l2_regularization'],
):
    model = HistGradientBoostingClassifier(
        learning_rate=lr, max_depth=MAX_DEPTH,
        min_samples_leaf=leaf, max_iter=n_iter,
        l2_regularization=l2, class_weight='balanced',
        random_state=42
    )
    cv_scores = []
    for tr_idx, val_idx in gkf.split(X_tr, y_tr, groups):
        model.fit(X_tr[tr_idx], y_tr[tr_idx])
        pred = model.predict(X_tr[val_idx])
        cv_scores.append(f1_score(y_tr[val_idx], pred, average='macro', zero_division=0))

    cv_mean = round(np.mean(cv_scores), 4)
    cv_std  = round(np.std(cv_scores), 4)
    results.append({'lr': lr, 'leaf': leaf, 'max_iter': n_iter,
                    'l2': l2, 'cv_f1': cv_mean, 'cv_std': cv_std})
    print(f'lr={lr}  leaf={leaf:>3}  iter={n_iter}  l2={l2} | CV: {cv_mean:.3f} ± {cv_std:.3f}')

result_df = pd.DataFrame(results).sort_values('cv_f1', ascending=False)
print('\n── CV Top 10 ──')
print(result_df.head(10).to_string(index=False))

Exception in thread Thread-3 (_readerthread):
Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\Lib\threading.py", line 1043, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "c:\ProgramData\anaconda3\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "c:\ProgramData\anaconda3\Lib\threading.py", line 994, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 1615, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "<frozen codecs>", line 325, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 24: invalid start byte
  File "c:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 262, in _count_physical_cores
    cpu_info = cpu_info.stdout.splitlines()
               ^^^^^^^^^^^^^^^^^

lr=0.01  leaf= 20  iter=100  l2=0.0 | CV: 0.297 ± 0.021
lr=0.01  leaf= 20  iter=100  l2=0.5 | CV: 0.290 ± 0.022
lr=0.01  leaf= 20  iter=100  l2=1.0 | CV: 0.295 ± 0.019
lr=0.01  leaf= 20  iter=200  l2=0.0 | CV: 0.333 ± 0.021
lr=0.01  leaf= 20  iter=200  l2=0.5 | CV: 0.329 ± 0.021
lr=0.01  leaf= 20  iter=200  l2=1.0 | CV: 0.334 ± 0.022


In [ ]:
# ── 최적 설정으로 Holdout 단 1회 평가 ────────────────
best = result_df.iloc[0]
print(f'최적 설정: lr={best.lr}  depth={MAX_DEPTH}  leaf={int(best.leaf)}  iter={int(best.max_iter)}  l2={best.l2}')
print(f'CV F1: {best.cv_f1:.3f} ± {best.cv_std:.3f}')

best_model = HistGradientBoostingClassifier(
    learning_rate=best.lr, max_depth=MAX_DEPTH,
    min_samples_leaf=int(best.leaf), max_iter=int(best.max_iter),
    l2_regularization=best.l2, class_weight='balanced',
    random_state=42
)
best_model.fit(X_tr, y_tr)
ho_pred = best_model.predict(X_ho)
ho_f1   = f1_score(y_ho, ho_pred, average='macro', zero_division=0)

print(f'\nHoldout macro-F1: {ho_f1:.4f}')
print('\n── 클래스별 성능 ──')
print(classification_report(y_ho, ho_pred, target_names=['down','neutral','up'], zero_division=0))

In [ ]:
# ── v1 vs v2 비교 ─────────────────────────────────────
print('── 튜닝 버전 비교 ──')
print(f'v1 (lr×depth×leaf 27가지)  → Holdout F1: 0.3971')
print(f'v2 (+ max_iter×l2  81가지) → Holdout F1: {ho_f1:.4f}')

result_df.to_csv(DATA_DIR + 'tune_v2_results.csv', index=False)
print('\n→ tune_v2_results.csv 저장 완료')